# Asset Sirius — Teoria das Filas aplicada ao atendimento de resgates

## Objetivo

Transformar a previsao de resgates em uma decisao operacional: **quantos gerentes sao necessarios para atender os alertas no tempo certo e aumentar a chance de retencao do cotista?**

## Como a teoria se encaixa no projeto

```text
Dados CVM + movimentacoes
          ↓
Deteccao de risco/resgate
          ↓
Alertas de retencao -> Fila FIFO -> Gerentes responsaveis
                                      ↓
                             Contato com o cotista
```

| Elemento da fila | Aplicacao no Asset Sirius |
|---|---|
| **lambda (λ)** | Alertas de resgate gerados por minuto |
| **mu (μ)** | Alertas tratados por minuto por gerente |
| **c (servidores)** | Numero de gerentes disponiveis |
| **Fila** | Alertas aguardando atendimento |
| **Disciplina** | FIFO: primeiro alerta que chega, primeiro atendido |
| **SLA** | Tempo maximo aceitavel de espera |

## Modelo utilizado: M/M/c

O modelo M/M/c representa chegadas aleatorias, tempo de atendimento variavel e varios gerentes atendendo uma fila compartilhada. Com a formula de **Erlang C**, estimamos utilizacao, probabilidade de espera, tamanho da fila e tempo de atendimento.

A **Lei de Little** conecta os principais indicadores:

```text
L = lambda x W
```

- `L`: quantidade media de alertas no sistema;
- `lambda`: taxa media de chegada;
- `W`: tempo medio do alerta no sistema.

## Decisoes apoiadas

1. Encontrar a quantidade minima de gerentes para cumprir o SLA;
2. Simular o impacto do aumento de resgates;
3. Comparar uma ou mais capacidades de atendimento;
4. Avaliar custo de capacidade versus custo da espera;
5. Apoiar escala operacional, priorizacao e infraestrutura do sistema.

> **Mensagem principal:** a teoria das filas permite dimensionar a capacidade de retencao antes que a fila de alertas cresca e o contato com o cotista perca efetividade.

## Hipoteses e unidades

- `lambda`: alertas por minuto;
- `mu`: alertas tratados por minuto por gerente;
- `c`: numero de gerentes trabalhando em paralelo;
- disciplina FIFO;
- limite operacional do exemplo: espera media de ate cinco minutos;
- as taxas abaixo sao mockadas e devem ser substituidas por logs reais de abertura e encerramento dos atendimentos.

O modelo M/M/c e uma aproximacao: ele pressupoe chegadas Poisson e tempos de atendimento exponenciais. A simulacao ajuda a avaliar o efeito da variabilidade.

In [13]:
from dataclasses import dataclass
from math import factorial, inf
from random import Random

@dataclass
class MetricasFila:
    taxa_chegada: float
    taxa_atendimento: float
    servidores: int
    utilizacao: float
    probabilidade_espera: float
    fila_media: float
    sistema_medio: float
    espera_media: float
    tempo_medio_sistema: float

def calcular_mmc(taxa_chegada, taxa_atendimento, servidores):
    if taxa_chegada <= 0 or taxa_atendimento <= 0 or servidores < 1:
        raise ValueError('Taxas devem ser positivas e servidores >= 1.')

    carga_ofertada = taxa_chegada / taxa_atendimento
    utilizacao = carga_ofertada / servidores
    if utilizacao >= 1:
        return MetricasFila(
            taxa_chegada, taxa_atendimento, servidores, utilizacao,
            1.0, inf, inf, inf, inf
        )

    probabilidade_vazio = 1 / (
        sum(carga_ofertada**n / factorial(n) for n in range(servidores))
        + carga_ofertada**servidores
        / (factorial(servidores) * (1 - utilizacao))
    )
    probabilidade_espera = (
        carga_ofertada**servidores
        / (factorial(servidores) * (1 - utilizacao))
        * probabilidade_vazio
    )
    espera_media = probabilidade_espera / (
        servidores * taxa_atendimento - taxa_chegada
    )
    tempo_medio_sistema = espera_media + 1 / taxa_atendimento
    return MetricasFila(
        taxa_chegada, taxa_atendimento, servidores, utilizacao,
        probabilidade_espera,
        taxa_chegada * espera_media,
        taxa_chegada * tempo_medio_sistema,
        espera_media,
        tempo_medio_sistema,
    )
def exibir(metricas):
    return {
        'servidores': metricas.servidores,
        'utilizacao': round(metricas.utilizacao, 3),
        'probabilidade_espera': round(metricas.probabilidade_espera, 3),
        'fila_media_Lq': round(metricas.fila_media, 3),
        'sistema_medio_L': round(metricas.sistema_medio, 3),
        'espera_media_Wq_min': round(metricas.espera_media, 3),
        'tempo_medio_W_min': round(metricas.tempo_medio_sistema, 3),
    }

## Exemplo:

Agora usamos variaveis ficticias para representar alertas de resgate por segmento. Os valores sao apenas demonstrativos:

| Segmento | lambda: alertas/min | mu: alertas/min por gerente |
|---|---:|---:|
| Varejo | 0,10 | 0,20 |
| Alta Renda | 0,06 | 0,15 |
| Private | 0,025 | 0,10 |

O SLA adotado e de no maximo cinco minutos de espera media. O calculo mostra o que ocorre com um e dois gerentes.

In [14]:
segmentos_mockados = {
    'Varejo': {'lambda': 0.10, 'mu': 0.20},
    'Alta Renda': {'lambda': 0.06, 'mu': 0.15},
    'Private': {'lambda': 0.025, 'mu': 0.10},
}
limite_sla_minutos = 5
for nome, parametros in segmentos_mockados.items():
    um_gerente = calcular_mmc(parametros['lambda'], parametros['mu'], 1)
    dois_gerentes = calcular_mmc(parametros['lambda'], parametros['mu'], 2)
    print(f'\n{nome}')
    print('  1 gerente:', exibir(um_gerente))
    print('  2 gerentes:', exibir(dois_gerentes))
    print('  1 gerente cumpre SLA:', um_gerente.espera_media <= limite_sla_minutos)


Varejo
  1 gerente: {'servidores': 1, 'utilizacao': 0.5, 'probabilidade_espera': 0.5, 'fila_media_Lq': 0.5, 'sistema_medio_L': 1.0, 'espera_media_Wq_min': 5.0, 'tempo_medio_W_min': 10.0}
  2 gerentes: {'servidores': 2, 'utilizacao': 0.25, 'probabilidade_espera': 0.1, 'fila_media_Lq': 0.033, 'sistema_medio_L': 0.533, 'espera_media_Wq_min': 0.333, 'tempo_medio_W_min': 5.333}
  1 gerente cumpre SLA: True

Alta Renda
  1 gerente: {'servidores': 1, 'utilizacao': 0.4, 'probabilidade_espera': 0.4, 'fila_media_Lq': 0.267, 'sistema_medio_L': 0.667, 'espera_media_Wq_min': 4.444, 'tempo_medio_W_min': 11.111}
  2 gerentes: {'servidores': 2, 'utilizacao': 0.2, 'probabilidade_espera': 0.067, 'fila_media_Lq': 0.017, 'sistema_medio_L': 0.417, 'espera_media_Wq_min': 0.278, 'tempo_medio_W_min': 6.944}
  1 gerente cumpre SLA: True

Private
  1 gerente: {'servidores': 1, 'utilizacao': 0.25, 'probabilidade_espera': 0.25, 'fila_media_Lq': 0.083, 'sistema_medio_L': 0.333, 'espera_media_Wq_min': 3.333, 'temp

### Interpretacao do exemplo mockado

- Varejo apresenta a maior taxa de chegada e fica no limite do SLA com um gerente;
- Alta Renda e Private ficam abaixo do limite com um gerente, mas ainda possuem espera operacional;
- dois gerentes reduzem a espera de forma desproporcional, criando folga para picos de resgates;
- em producao, as taxas devem ser calculadas a partir dos eventos reais do pipeline e dos registros de atendimento.

A decisao final deve combinar o SLA, o custo de cada gerente, o valor sob risco e a prioridade do alerta.

## Programacao inteira para capacidade

A teoria das filas informa o desempenho de cada capacidade. A programacao inteira pode escolher a configuracao com menor custo total:

$$\min \sum_s custo \cdot c_s + penalidade \cdot lambda_s \cdot Wq_s \cdot 60$$

Sujeito a:

- `c_s` inteiro e positivo para cada segmento;
- `Wq_s <= SLA`;
- limite de gerentes e orçamento disponíveis.

Assim, a programação inteira complementa a teoria das filas: a primeira escolhe a capacidade e a segunda mede a espera, a utilização e o risco operacional resultantes.